# Architecture Diagram

## End-to-end data flow

```{mermaid}
flowchart LR
  R["huggingface/optimum<br/>59 .py files"] --> PS["Parser Service<br/>ast to CPG, stable ids"]
  PS --> T1["cpg.nodes"]
  PS --> T2["cpg.edges"]
  PS --> T3["cpg.metadata"]
  PS --> T4["cpg.errors"]
  T1 --> NC["Neo4j Kafka Connector Sink<br/>Cypher MERGE"]
  T2 --> NC
  NC --> NEO[("Neo4j<br/>CpgNode / CPG_EDGE")]
  T3 --> SP["Spark Structured Streaming<br/>foreachBatch upsert + checkpoint"]
  SP --> MG[("MongoDB<br/>file_metadata")]
```

## Component responsibilities

| Component | Responsibility | Idempotency contribution |
|---|---|---|
| Parser Service | one file at a time, bounded memory | structural ids, generation marker |
| Kafka | decouples producer from two independent sinks | stable id as key, log compaction |
| Neo4j Connector Sink | graph topology, no Spark in the path | `MERGE` on stable id |
| Spark Structured Streaming | metadata transformation and delivery | offset checkpoint |
| MongoDB Spark Connector | document persistence | upsert on `file_id` |
| Generation sweep | removes elements deleted by an edit | scoped delete by `file_hash` |

## Why the two branches differ

The lab prescribes different ingestion paths, and the reason is architectural
rather than arbitrary. Graph topology is a **high-volume, low-transformation**
stream: tens of thousands of small writes whose only requirement is idempotent
upsert, which Cypher expresses directly — inserting Spark would add latency and a
failure domain for no benefit. Metadata is a **low-volume, higher-transformation**
stream: 59 documents that benefit from schema enforcement and aggregation, which
is exactly what Structured Streaming provides.

## Rendering this diagram

The Mermaid block above renders in the published book once
`sphinxcontrib-mermaid` is enabled (already configured in `_config.yml`). The
source is also kept standalone at `config/architecture.mmd`.